# [Ablation Study] Benchmark Đầu Nháp EAGLE-3.1 Cho Qwen2.5-7B Trên GPU Tesla T4 (Kaggle)

Notebook này phục vụ bài đo đối chứng nâng cao (**Ablation Study**) trong **Chương 5** của Đồ án Tốt nghiệp:
- **Target Model**: `Qwen/Qwen2.5-7B-Instruct-AWQ` (W4A16 AutoAWQ Marlin, sm_75)
- **Draft Model mới**: `duckhoi/qwen2.5-7b-eagle3.1-speculator` (EAGLE-3.1 Draft Head tự huấn luyện, k=3)
- **Baseline so sánh**: Số liệu chuẩn của Làn 1 (Vanilla) và Làn 2 (`Qwen2.5-0.5B-Instruct`, k=3) từ Bảng 4.2 của Đồ án.
- **Tập Workload**: Giữ nguyên 100% 3 bộ workload chuẩn hóa từ Sprint 1-2:
  - **W1 (VAT JSON Extraction)**: 32 requests hóa đơn MC-OCR (`experiments/configs/w1/node1_chat_requests.jsonl`, max_tokens=256)
  - **W2 (Text-to-SQL)**: 32 requests kế toán SQLite (`experiments/configs/w2_v2/chat_requests.jsonl`, max_tokens=96)
  - **W3 (Free Analysis)**: 32 requests báo cáo tài chính tự do (`experiments/configs/w3_v2/chat_requests.jsonl`, max_tokens=192)

In [ ]:
# 1. Kiểm tra phần cứng GPU Tesla T4 trên Kaggle
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")

In [ ]:
# 2. Cài đặt các thư viện cần thiết và chuẩn bị Repo Đồ Án (Hỗ trợ Private Repo)
import os
import sys
import subprocess

# Tắt HF Transfer để tránh xung đột môi trường
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Cài đặt vLLM và các công cụ đo kiểm
!pip install -q -U "vllm>=0.6.0" "nvidia-ml-py" "aiohttp" "tabulate" "pynvml" "pandas"

# Chuẩn bị thư mục repo
REPO_DIR = "/kaggle/working/do-an-ptit-d22"

if not os.path.exists(REPO_DIR):
    # Cách 1: Kiểm tra xem đã upload repo qua Kaggle Dataset chưa
    dataset_candidates = ["/kaggle/input/do-an-ptit-d22", "/kaggle/input/doanptitd22"]
    matched_input = next((p for p in dataset_candidates if os.path.exists(p)), None)
    if matched_input:
        print(f"📦 Tìm thấy repo đính kèm từ Kaggle Dataset: {matched_input}")
        !cp -r {matched_input} {REPO_DIR}
    else:
        # Cách 2: Clone qua GitHub Personal Access Token (PAT)
        gh_token = None
        try:
            from kaggle_secrets import UserSecretsClient
            user_secrets = UserSecretsClient()
            gh_token = user_secrets.get_secret("GH_TOKEN") or user_secrets.get_secret("GITHUB_TOKEN")
            print("🔑 Đã tự động lấy GH_TOKEN từ Kaggle Secrets!")
        except Exception:
            pass
        
        if not gh_token:
            import getpass
            print("ℹ️ Repo private. Vui lòng nhập GitHub PAT (hoặc cấu hình secret 'GH_TOKEN' tại Add-ons -> Secrets):")
            gh_token = getpass.getpass("Nhập GitHub Token (ký tự bị ẩn): ").strip()
        
        clone_url = f"https://{gh_token}@github.com/theflyingkhui04/do-an-ptit-d22.git"
        ret = subprocess.run(["git", "clone", clone_url, REPO_DIR], capture_output=True, text=True)
        if ret.returncode == 0:
            print(f"✅ Clone thành công repo vào {REPO_DIR}")
        else:
            err_msg = ret.stderr.replace(gh_token, "***") if gh_token else ret.stderr
            print(f"❌ Lỗi khi clone repo: {err_msg}")
else:
    print(f"✅ Repo đã tồn tại tại {REPO_DIR}")

%cd {REPO_DIR}
print("📂 Thư mục làm việc hiện tại:", os.getcwd())

In [ ]:
# 3. Khởi chạy vLLM Server chạy ngầm với Target 7B AWQ + Speculator EAGLE-3.1 (k=3)
import subprocess
import time
import requests
import json

PORT = 8004
LOG_FILE = "/kaggle/working/vllm_eagle3_server.log"

print(f"Đang khởi động vLLM Server trên cổng {PORT}...")
print("Target: Qwen/Qwen2.5-7B-Instruct-AWQ | Draft: duckhoi/qwen2.5-7b-eagle3.1-speculator (k=3)")

# Dọn dẹp tiến trình cũ nếu có
!pkill -f "vllm" || true
time.sleep(2)

cmd = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "Qwen/Qwen2.5-7B-Instruct-AWQ",
    "--host", "127.0.0.1",
    "--port", str(PORT),
    "--served-model-name", "qwen2.5-7b-eagle3-draft",
    "--generation-config", "vllm",
    "--quantization", "awq",
    "--dtype", "half",
    "--gpu-memory-utilization", "0.85",
    "--max-model-len", "2048",
    "--max-num-seqs", "32",
    "--speculative-model", "duckhoi/qwen2.5-7b-eagle3.1-speculator",
    "--num-speculative-tokens", "3",
    "--per-request-spec-decode-metrics", "summary"
]

server_process = subprocess.Popen(
    cmd,
    stdout=open(LOG_FILE, "w"),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid
)

# Chờ server sẵn sàng (poll /health)
start_time = time.time()
server_ready = False
for _ in range(90):
    try:
        r = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=3)
        if r.status_code == 200:
            server_ready = True
            break
    except Exception:
        pass
    time.sleep(5)
    print(f"Đang chờ vLLM khởi động... ({int(time.time() - start_time)}s)")

if server_ready:
    print(f"✅ vLLM Server sẵn sàng sau {int(time.time() - start_time)} giây!")
else:
    print("❌ Lỗi: Server khởi động thất bại hoặc quá thời gian. Xem log dưới đây:")
    !tail -n 40 {LOG_FILE}

In [ ]:
# 4. Thực thi Benchmark Harness đo đạc qua 3 Workload chuẩn W1, W2, W3
# Khớp 100% giao thức thực nghiệm Sprint 1-2 của đồ án
import subprocess
import os

ENDPOINT = f"http://127.0.0.1:{PORT}/v1/chat/completions"
RAW_DIR = "/kaggle/working/benchmark_runs"
os.makedirs(RAW_DIR, exist_ok=True)

# Cấu hình quét Concurrency: [1, 2, 4, 8, 16, 32]
CONCURRENCY_SWEEP = "1 2 4 8 16 32"
REPEATS_PER_CELL = 3  # Lặp 3 lần mỗi cell để lấy median p50 tin cậy

WORKLOADS = [
    {
        "id": "W1",
        "file": "experiments/configs/w1/node1_chat_requests.jsonl",
        "name": "VAT JSON Extraction (W1)"
    },
    {
        "id": "W2",
        "file": "experiments/configs/w2_v2/chat_requests.jsonl",
        "name": "Text-to-SQL (W2)"
    },
    {
        "id": "W3",
        "file": "experiments/configs/w3_v2/chat_requests.jsonl",
        "name": "Free Analysis (W3)"
    }
]

for w in WORKLOADS:
    print(f"\n=======================================================")
    print(f"🚀 BẮT ĐẦU ĐO WORKLOAD: {w['name']}")
    print(f"📁 File requests: {w['file']}")
    print(f"⚙️ Concurrency sweep: {CONCURRENCY_SWEEP}")
    print(f"=======================================================")
    
    raw_out = f"{RAW_DIR}/raw_eagle3_{w['id']}.jsonl"
    summary_csv = f"{RAW_DIR}/summary_eagle3_{w['id']}.csv"
    
    cmd = [
        "python3", "experiments/benchmark_harness.py",
        "--endpoint", ENDPOINT,
        "--workload-file", w["file"],
        "--workload", w["id"],
        "--lane", "static",
        "--concurrency-sweep", *CONCURRENCY_SWEEP.split(),
        "--repeats", str(REPEATS_PER_CELL),
        "--warmup-requests", "3",
        "--raw-out", raw_out,
        "--summary-csv", summary_csv,
        "--overwrite",
        "--skip-env-check"
    ]
    
    ret = subprocess.run(cmd)
    if ret.returncode != 0:
        print(f"❌ Lỗi khi chạy benchmark cho {w['id']}! Exit code: {ret.returncode}")
    else:
        print(f"✅ Hoàn tất đo Workload {w['id']}. Kết quả lưu tại: {summary_csv}")

In [ ]:
# 5. Tổng hợp Kết quả & Đối chiếu Trực diện với Baseline 0.5B (Bảng 4.2 của Đồ án)
import pandas as pd
import glob
import os
from tabulate import tabulate

# Dữ liệu đối chứng chuẩn của Sprint 1 (M1 Baseline - Bảng 4.2 trong docs/chuong-4-thuc-nghiem.md)
BASELINE_DATA = {
    ("W1", 1):  {"vanilla_tp": 36.38,  "draft05_tp": 48.48,  "draft05_alpha": 91.59, "draft05_speedup": 1.34},
    ("W1", 2):  {"vanilla_tp": 66.81,  "draft05_tp": 85.16,  "draft05_alpha": 91.62, "draft05_speedup": 1.29},
    ("W1", 4):  {"vanilla_tp": 112.02, "draft05_tp": 147.21, "draft05_alpha": 91.63, "draft05_speedup": 1.33},
    ("W1", 8):  {"vanilla_tp": 172.91, "draft05_tp": 214.64, "draft05_alpha": 91.66, "draft05_speedup": 1.26},
    ("W1", 16): {"vanilla_tp": 224.55, "draft05_tp": 271.91, "draft05_alpha": 91.65, "draft05_speedup": 1.22},
    ("W1", 32): {"vanilla_tp": 264.65, "draft05_tp": 304.04, "draft05_alpha": 91.57, "draft05_speedup": 1.16},
    
    ("W2", 1):  {"vanilla_tp": 39.21,  "draft05_tp": 48.87,  "draft05_alpha": 80.30, "draft05_speedup": 1.34},
    ("W2", 2):  {"vanilla_tp": 74.55,  "draft05_tp": 89.99,  "draft05_alpha": 80.24, "draft05_speedup": 1.32},
    ("W2", 4):  {"vanilla_tp": 138.94, "draft05_tp": 169.73, "draft05_alpha": 81.21, "draft05_speedup": 1.33},
    ("W2", 8):  {"vanilla_tp": 232.30, "draft05_tp": 263.54, "draft05_alpha": 80.24, "draft05_speedup": 1.23},
    ("W2", 16): {"vanilla_tp": 357.47, "draft05_tp": 383.12, "draft05_alpha": 80.74, "draft05_speedup": 1.17},
    ("W2", 32): {"vanilla_tp": 491.43, "draft05_tp": 481.22, "draft05_alpha": 80.82, "draft05_speedup": 1.04},
    
    ("W3", 1):  {"vanilla_tp": 39.24,  "draft05_tp": 39.25,  "draft05_alpha": 54.62, "draft05_speedup": 1.00},
    ("W3", 2):  {"vanilla_tp": 76.18,  "draft05_tp": 73.72,  "draft05_alpha": 54.37, "draft05_speedup": 0.98},
    ("W3", 4):  {"vanilla_tp": 139.87, "draft05_tp": 141.90, "draft05_alpha": 55.53, "draft05_speedup": 1.00},
    ("W3", 8):  {"vanilla_tp": 241.40, "draft05_tp": 225.96, "draft05_alpha": 55.18, "draft05_speedup": 0.93},
    ("W3", 16): {"vanilla_tp": 351.13, "draft05_tp": 322.08, "draft05_alpha": 54.28, "draft05_speedup": 0.90},
    ("W3", 32): {"vanilla_tp": 483.10, "draft05_tp": 394.99, "draft05_alpha": 53.95, "draft05_speedup": 0.81},
}

summary_files = glob.glob("/kaggle/working/benchmark_runs/summary_eagle3_*.csv")
comparison_rows = []

if summary_files:
    df_new = pd.concat([pd.read_csv(f) for f in summary_files], ignore_index=True)
    
    for idx, row in df_new.iterrows():
        w = str(row.get("workload", "")).upper()
        c = int(row.get("concurrency", 1))
        
        eagle_tp = float(row.get("throughput", 0.0))
        eagle_alpha = float(row.get("alpha", 0.0))
        eagle_tpot = float(row.get("tpot_p50", 0.0))
        
        base = BASELINE_DATA.get((w, c), {})
        vanilla_tp = base.get("vanilla_tp", 0.0)
        draft05_tp = base.get("draft05_tp", 0.0)
        draft05_alpha = base.get("draft05_alpha", 0.0)
        draft05_speedup = base.get("draft05_speedup", 1.0)
        
        eagle_speedup = round(eagle_tp / vanilla_tp, 2) if vanilla_tp > 0 else 0.0
        gain_vs_draft05 = round(((eagle_tp - draft05_tp) / draft05_tp) * 100, 1) if draft05_tp > 0 else 0.0
        
        comparison_rows.append({
            "Workload": w,
            "Concurrency": c,
            "Vanilla (tok/s)": vanilla_tp,
            "Draft 0.5B (tok/s)": draft05_tp,
            "EAGLE-3.1 (tok/s)": round(eagle_tp, 2),
            "Draft 0.5B Alpha (%)": f"{draft05_alpha:.1f}%",
            "EAGLE-3.1 Alpha (%)": f"{eagle_alpha:.1f}%",
            "Speedup 0.5B": f"{draft05_speedup}x",
            "Speedup EAGLE-3.1": f"{eagle_speedup}x",
            "Delta vs 0.5B (%)": f"{gain_vs_draft05:+.1f}%"
        })

if comparison_rows:
    df_cmp = pd.DataFrame(comparison_rows)
    print("\n================ BẢNG ĐỐI SOÁT TRỰC DIỆN: EAGLE-3.1 VS DRAFT 0.5B ================\n")
    print(tabulate(df_cmp, headers="keys", tablefmt="pipe", showindex=False))
    
    out_csv = "/kaggle/working/eagle3_vs_draft05_comparison.csv"
    df_cmp.to_csv(out_csv, index=False)
    print(f"\n💾 Bảng kết quả đối chứng đã được lưu tại: {out_csv}")
else:
    print("Chưa tìm thấy file summary để đối soát.")

In [ ]:
# 6. Dọn dẹp tiến trình server và giải phóng VRAM GPU
import os
import signal

print("Đang giải phóng GPU...")
try:
    os.killpg(os.getpgid(server_process.pid), signal.SIGTERM)
    print("Đã dừng tiến trình vLLM Server.")
except Exception as e:
    print(f"Lỗi khi dừng server: {e}")

!pkill -f "vllm" || true
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ GPU đã được giải phóng sạch sẽ.")